In [13]:
import pandas as pd
import numpy as np
df = pd.read_pickle('pig_features_data_all.pkl')

In [3]:
import os
import re

# 配置目录路径
# 注意：如果路径中包含空格或特殊字符，确保字符串前加 r 或者使用双反斜杠
folder_path = r"C:\Users\shamo\Desktop\无猪 被误判为 有猪"

def extract_unique_ids(path):
    unique_ids = set()  # 使用集合自动去重
    
    # 检查目录是否存在
    if not os.path.exists(path):
        print(f"错误：找不到目录 {path}")
        return []

    # 遍历目录下的所有文件
    for filename in os.listdir(path):
        # 只处理 .html 文件
        if filename.lower().endswith('.html'):
            # 使用正则提取开头的数字编号
            # 逻辑：匹配字符串开头 (^) 的一个或多个数字 (\d+)
            match = re.match(r'^(\d+)', filename)
            
            if match:
                code = match.group(1)
                unique_ids.add(code)
    
    # 将集合转换为列表并排序（方便查看）
    result_list = sorted(list(unique_ids))
    return result_list

In [4]:
ids_list = extract_unique_ids(folder_path)

In [5]:
ids_list

['19317', '19829', '29509', '29524', '30006', '30029', '30305']

In [14]:
def filter_dataframe(df, ids_list):
    if not ids_list:
        print("编号列表为空，返回原始数据。")
        return df

    # 1. 提取文件名：获取 file_path 中最后一个 '/' 之后的部分
    # 如果路径分隔符可能是 '\' (Windows)，可以使用 os.path.basename，但这里看数据是 Linux 风格 '/'
    file_names = df['file_path'].apply(lambda x: x.split('/')[-1])
    
    # 2. 构建掩码 (Mask)
    # 逻辑：检查文件名是否以 ids_list 中的任意一个字符串开头
    # startswith 支持传入一个元组 (tuple)，会检查是否以元组中任意一个元素开头
    mask_to_remove = file_names.str.startswith(tuple(ids_list))
    print(np.sum(mask_to_remove))
    
    # 3. 取反 (~) 以保留不需要删除的行
    df_filtered = df[~mask_to_remove].reset_index(drop=True)
    
    return df_filtered


In [18]:
filtered_df = filter_dataframe(df, ids_list)

689


In [19]:
filtered_df.to_pickle('pig_features_data_all_filtered.pkl')